# Neutral Threshold Robustness Analysis

Robustness check: does the global peak
disappearance at intermediate aztreonam concentrations persist under
reasonable perturbations of the neutrality cutoff?

1. **Check graphs exist** — verify all expected GraphML files were built
2. **Analyze** — load each graph, check for global peak existence
3. **Visualize** — heatmap + by-concentration summary plot

## 1. Check Graphs Exist

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

# Nature journal style
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "legend.fontsize": 6,
    "axes.linewidth": 0.5,
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "lines.linewidth": 1.0,
    "lines.markersize": 4,
    "figure.dpi": 150,
    "savefig.dpi": 1000,
    "savefig.bbox": "tight",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})
sns.set_style("ticks")

# Nature column widths
SINGLE_COL = 89 / 25.4   # 89mm in inches
DOUBLE_COL = 183 / 25.4   # 183mm in inches

In [ ]:
import os
from pathlib import Path

# Neutral-threshold graph sweep produced by reproduce_s18.py (repo-relative;
# override with the S18_GRAPHS_DIR environment variable).
output_dir = Path(os.environ.get("S18_GRAPHS_DIR", "outputs/global-peak-robustness"))
graph_files = sorted(output_dir.glob("azt_c*_t*.graphml"))
expected_concentrations = [0.0, 0.44, 1.33, 4.0, 12.0, 36.0, 108.0, 324.0]

print(f"Found {len(graph_files)} GraphML files in {output_dir}")

if not graph_files:
    print("\nNo graph files found. Run the graph-building step first:")
    print("  python reproduce_s18.py")
else:
    # Parse filenames to summarize what's available
    thresholds = set()
    concentrations = set()
    for f in graph_files:
        name = f.stem
        conc_part, thresh_part = name.rsplit("_t", 1)
        conc_str = conc_part.split("_", 1)[1][1:]  # remove "azt_" then "c"
        concentrations.add(float(conc_str.replace("_", ".")))
        thresholds.add(float(thresh_part.replace("_", ".")))

    found_concs = sorted(concentrations)
    expected_concs = sorted(expected_concentrations)
    missing_concs = sorted(set(expected_concs) - set(found_concs))

    print(f"Concentrations: {found_concs}")
    print(f"Thresholds: {len(thresholds)} values, range [{min(thresholds):.2f}, {max(thresholds):.2f}]")
    print(f"Expected (all AZT): {len(expected_concs)} x {len(thresholds)} = {len(expected_concs) * len(thresholds)} graphs")

    if missing_concs:
        print(f"Missing concentrations: {missing_concs}")
    else:
        print("All AZT concentrations are present.")

## 2. Analyze Neutral Threshold Robustness

In [ ]:
import polars as pl
from fitness_landscape_graph.graph_analyzer import GraphAnalyzer

MIN_GROUP_SIZE = 12
results = []

for i, graph_path in enumerate(graph_files, 1):
    filename = graph_path.stem

    # Parse filename: azt_c{conc}_t{threshold}.graphml
    # Example: azt_c12_0_t0_15.graphml -> concentration=12.0, threshold=0.15
    try:
        conc_part, thresh_part = filename.rsplit("_t", 1)
        conc_str = conc_part.split("_", 1)[1][1:]  # remove "azt_" then "c"
        concentration = float(conc_str.replace("_", "."))
        neutral_threshold = float(thresh_part.replace("_", "."))
    except (IndexError, ValueError) as e:
        print(f"Warning: Could not parse {filename}: {e}")
        continue

    analyzer = GraphAnalyzer(str(graph_path))
    has_peak, node_id, info = analyzer.has_global_peak(min_group_size=MIN_GROUP_SIZE)

    results.append({
        "concentration": concentration,
        "neutral_threshold": neutral_threshold,
        "has_global_peak": has_peak,
        "peak_node": node_id if has_peak else None,
        "peak_fitness": info.get("fitness") if has_peak else None,
        "peak_group_size": info.get("group_size") if has_peak else None,
    })

    if i % 20 == 0 or i == len(graph_files):
        print(f"  Processed {i}/{len(graph_files)} graphs...")

df = pl.DataFrame(results).sort(["concentration", "neutral_threshold"])
print(f"\nCollected {len(df)} results")
df.head()

In [ ]:
# Summary statistics
total_with_peak = df["has_global_peak"].sum()
print(f"Total graphs: {len(df)}")
print(f"Graphs with global peak: {total_with_peak} ({100 * total_with_peak / len(df):.1f}%)")

print("\nBy concentration:")
conc_summary = (
    df.group_by("concentration")
    .agg([
        pl.col("has_global_peak").sum().alias("count_with_peak"),
        pl.col("has_global_peak").len().alias("total"),
    ])
    .sort("concentration")
)
for row in conc_summary.iter_rows(named=True):
    pct = 100 * row["count_with_peak"] / row["total"] if row["total"] > 0 else 0
    print(f"  {row['concentration']:6.1f}: {row['count_with_peak']:2d}/{row['total']:2d} ({pct:5.1f}%)")

print("\nBy threshold range:")
for t_min, t_max, label in [(0.15, 0.24, "0.15-0.24"), (0.25, 0.34, "0.25-0.34"), (0.35, 0.45, "0.35-0.45")]:
    subset = df.filter((pl.col("neutral_threshold") >= t_min) & (pl.col("neutral_threshold") <= t_max))
    count = subset["has_global_peak"].sum()
    total = len(subset)
    pct = 100 * count / total if total > 0 else 0
    print(f"  {label}: {count:2d}/{total:2d} ({pct:5.1f}%)")

In [ ]:
# Save CSV
csv_path = output_dir / f"global_peak_analysis_{MIN_GROUP_SIZE}.csv"
df.write_csv(csv_path)
print(f"Saved to {csv_path}")

## 3. Visualization

In [ ]:
import numpy as np
from matplotlib.colors import ListedColormap

# --- Heatmap: threshold x concentration ---
concentrations = sorted(df["concentration"].unique().to_list())
thresholds_list = sorted(df["neutral_threshold"].unique().to_list())

# Build 2D array [concentrations x thresholds]
heatmap_data = np.zeros((len(concentrations), len(thresholds_list)))
for row in df.iter_rows(named=True):
    i = concentrations.index(row["concentration"])
    j = thresholds_list.index(row["neutral_threshold"])
    heatmap_data[i, j] = 1 if row["has_global_peak"] else 0

# Annotation array with checkmarks/crosses
annot_array = np.where(heatmap_data == 1, "\u2713", "\u2717")

fig, ax = plt.subplots(figsize=(DOUBLE_COL, SINGLE_COL * 0.7))

cmap = ListedColormap(["#d9d9d9", "#2ca02c"])

sns.heatmap(
    heatmap_data,
    ax=ax,
    cmap=cmap,
    vmin=0,
    vmax=1,
    annot=annot_array,
    fmt="s",
    annot_kws={"size": 5, "ha": "center", "va": "center"},
    linewidths=0.3,
    linecolor="white",
    cbar=False,
    xticklabels=[f"{t:.2f}" if i % 3 == 0 else "" for i, t in enumerate(thresholds_list)],
    yticklabels=[f"{c:g}" for c in concentrations],
)

ax.set_xlabel("Neutral threshold")
ax.set_ylabel("Concentration (\u00b5g/mL)")

# Add legend in the empty space above the heatmap (upper right area)
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#2ca02c", edgecolor="black", linewidth=0.3, label="Global peak present"),
    Patch(facecolor="#d9d9d9", edgecolor="black", linewidth=0.3, label="No global peak"),
]
ax.legend(handles=legend_elements, loc="lower right", bbox_to_anchor=(1.0, 1.01), frameon=False, fontsize=6)

plt.tight_layout()

heatmap_path = output_dir / "global_peak_heatmap.png"
fig.savefig(heatmap_path, bbox_inches="tight")
fig.savefig(output_dir / f"global_peak_heatmap_{MIN_GROUP_SIZE}.png", bbox_inches="tight", dpi=1000)
print(f"Saved heatmap to {heatmap_path}")
plt.show()

In [ ]:
# --- By-concentration plot ---
summary = (
    df.group_by("concentration")
    .agg([
        pl.col("has_global_peak").mean().alias("fraction_with_peak"),
        pl.col("has_global_peak").sum().alias("count_with_peak"),
        pl.col("has_global_peak").len().alias("total"),
    ])
    .sort("concentration")
)

concs = summary["concentration"].to_list()
fracs = summary["fraction_with_peak"].to_list()
counts = summary["count_with_peak"].to_list()
totals = summary["total"].to_list()

fig, ax = plt.subplots(figsize=(SINGLE_COL, SINGLE_COL * 0.8))
ax.plot(concs, fracs, marker="o", markersize=4, linewidth=1.0, color="#2E86AB")

for c, f, count, total in zip(concs, fracs, counts, totals):
    ax.annotate(
        f"{count}/{total}",
        xy=(c, f), xytext=(0, 6), textcoords="offset points",
        ha="center", fontsize=5,
    )

ax.set_xlabel("Concentration (\u00b5g/mL)")
ax.set_ylabel("Fraction of thresholds\nwith global peak")
ax.set_ylim(-0.05, 1.05)

# Explicitly set x-tick labels to show concentration values
ax.set_xticks(concs)
ax.set_xticklabels([f"{c:g}" for c in concs])

sns.despine(ax=ax)
plt.tight_layout()

conc_plot_path = output_dir / "global_peak_by_concentration.png"
fig.savefig(conc_plot_path, bbox_inches="tight")
fig.savefig(output_dir / "global_peak_by_concentration.pdf", bbox_inches="tight")
print(f"Saved to {conc_plot_path}")
plt.show()

In [ ]:
# --- By-threshold plot ---
summary_t = (
    df.group_by("neutral_threshold")
    .agg([
        pl.col("has_global_peak").mean().alias("fraction_with_peak"),
        pl.col("has_global_peak").sum().alias("count_with_peak"),
        pl.col("has_global_peak").len().alias("total"),
    ])
    .sort("neutral_threshold")
)

thresh_vals = summary_t["neutral_threshold"].to_list()
fracs_t = summary_t["fraction_with_peak"].to_list()

fig, ax = plt.subplots(figsize=(SINGLE_COL, SINGLE_COL * 0.8))
ax.plot(thresh_vals, fracs_t, marker="o", markersize=4, linewidth=1.0, color="#A23B72")

# Subtle vertical lines to demarcate threshold ranges instead of axvspan
for boundary in [0.245, 0.345]:
    ax.axvline(x=boundary, color="grey", linestyle=":", linewidth=0.4, alpha=0.6)

# Range labels at top
for xpos, label in [(0.195, "Low"), (0.295, "Mid"), (0.395, "High")]:
    ax.text(xpos, 1.02, label, ha="center", va="bottom", fontsize=5, color="grey",
            transform=ax.get_xaxis_transform())

ax.set_xlabel("Neutral threshold")
ax.set_ylabel("Fraction of concentrations\nwith global peak")
ax.set_ylim(-0.05, 1.05)
ax.set_xlim(0.14, 0.46)

sns.despine(ax=ax)
plt.tight_layout()

thresh_plot_path = output_dir / "global_peak_by_threshold.png"
fig.savefig(thresh_plot_path, bbox_inches="tight")
fig.savefig(output_dir / "global_peak_by_threshold.pdf", bbox_inches="tight")
print(f"Saved to {thresh_plot_path}")
plt.show()